# **Sales Analyst Agent**

## **About the Scenario**
In this scenario, we analyze sales data for AdventureWorks using Azure OpenAI. The AI agent performs tasks such as file retrieval, sales calculations, and generating insights by leveraging tools like File Search. This showcases how businesses can employ Azure OpenAI Agents to streamline analytical workflows and decision making.

Key Steps:

1. *File Conversion*: Converting Excel data into a Markdown format compatible with the AI agent.
2. *File Upload*: Storing the converted files in the Azure OpenAI Project for efficient processing.
3. *AI-Powered Analysis*: Leveraging Azure OpenAI to generate insights such as revenue metrics by region.

This hands-on demonstration will prepare you to use Azure OpenAI Assistants for similar real-world data engineering and analytics tasks.

## **Data**
This scenario uses files from the folder [`data/`](./data/) in this repo. You can clone this repo or copy this folder to make sure you have access to these files when running the sample.

The sales data is stored in multiple Excel files located in the data/ directory. These files contain essential details to simulate sales orders, which the AI agent will analyze.

Ensure you have the following files ready in the data/ directory:

- SalesOrder_43659.xlsx
- SalesOrder_43661.xlsx
- SalesOrder_43662.xlsx
- SalesOrder_43665.xlsx

## **Time**
You should expect to spend 10-15 minutes building and running this scenario. 

## **Before you begin**

#### Step 1: Install required libraries
Installing dependencies directly within a Jupyter notebook is a good practice because it ensures that all required packages are installed in the correct versions, making the notebook self-contained and reproducible. This approach helps other users or collaborators to set up the environment quickly and avoid potential issues related to missing or incompatible packages.

In [ ]:
# Install the packages
%pip install -r ./requirements.txt

print("\nPackages installed successfully.")

#### Step 2: Setting up the environment
Before we begin, we need to load the necessary environment variables from a `.env` file. These variables include sensitive information such as API keys and endpoint URLs, which are crucial for running the code successfully.

Here’s what you need to do:
- Ensure your `.env` file is properly configured in the `.venv/.env` format. We have provided an template `.env` file, `.env.example` for your reference.
- Verify that all required secrets are included in the file before running the code.


The `.env` file must contain the following secrets:
- PROJECT_CONNECTION_STRING: URL to connect to the Azure OpenAI Project to access project resources.
- AZURE_OPENAI_DEPLOYMENT: The name of the Azure OpenAI model deployment.

Now, let’s load these variables and get started!

<code style="background:yellow;color:black">Note: Make sure to keep your `.env` file secure and avoid sharing it publicly. </code>

*For more information about leveraging Python Virtual Environments can be found [here](https://docs.python.org/3/library/venv.html).*

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Retrieve the secrets
__PROJECT_ENDPOINT = os.getenv("AZURE_AI_AGENT_ENDPOINT")
__AZURE_OPENAI_DEPLOYMENT = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT_NAME")

# Verify environment variables
if not all([__PROJECT_ENDPOINT, __AZURE_OPENAI_DEPLOYMENT]):
    raise EnvironmentError("One or more environment variables are missing. Please check the .env file.")
else:
    print("Environment variables loaded successfully.")

### Tracing (optional)
Enable lightweight OpenTelemetry tracing for key steps. Set environment variable `ENABLE_CONSOLE_TRACING=true` to also print spans to the notebook output.

In [ ]:
# Minimal OpenTelemetry tracing setup
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor, ConsoleSpanExporter
from azure.monitor.opentelemetry import configure_azure_monitor
from opentelemetry.instrumentation.openai_v2 import OpenAIInstrumentor
from contextlib import contextmanager
import os

# Reuse existing project_client if defined, otherwise create one briefly to fetch connection string
try:
    _pc = project_client
except NameError:
    _pc = None

if _pc is None:
    from azure.ai.projects import AIProjectClient
    from azure.identity import DefaultAzureCredential
    _pc = AIProjectClient(endpoint=os.environ["AZURE_AI_AGENT_ENDPOINT"], credential=DefaultAzureCredential())

# Configure Azure Monitor using the project's Application Insights connection string
conn = _pc.telemetry.get_application_insights_connection_string()
configure_azure_monitor(connection_string=conn)

# Optional console exporter for local dev if ENABLE_CONSOLE_TRACING=true
if os.getenv("ENABLE_CONSOLE_TRACING", "false").lower() == "true":
    try:
        trace.get_tracer_provider().add_span_processor(SimpleSpanProcessor(ConsoleSpanExporter()))
        print("Console tracing enabled")
    except Exception:
        # Fallback: set a basic provider just for console if needed
        tp = TracerProvider()
        tp.add_span_processor(SimpleSpanProcessor(ConsoleSpanExporter()))
        trace.set_tracer_provider(tp)
        print("Console tracing enabled (fallback provider)")

# Instrument OpenAI SDK so LLM calls are traced
OpenAIInstrumentor().instrument()

tracer = trace.get_tracer(__name__)

@contextmanager
def traced_span(span_name: str, **attrs):
    with tracer.start_as_current_span(span_name) as span:
        for k, v in attrs.items():
            try:
                span.set_attribute(k, v if v is None or isinstance(v, (int, float, bool)) else str(v)[:1024])
            except Exception:
                pass
        yield

## **Azure OpenAI Agent Setup**

### Step 1: Initializing the Azure AI Studio Project Client
First, we will initialize the Azure AI Studio Project client using Azure’s `DefaultAzureCredential` for authentication, allowing seamless integration with Azure resources. You will need to log into Azure using the Azure CLI.

In [ ]:
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

try:
    # Initialize the Azure AI Project client
    project_client = AIProjectClient(
        endpoint=os.environ["AZURE_AI_AGENT_ENDPOINT"],
        credential=DefaultAzureCredential()
    )

    print("Azure AI Studio client created successfully.")

except Exception as e:
    print(f"Error creating the project client: {e}")

### Step 2: Initializing the Azure Agent Client
Next, we’ll initialize the Azure Agent Runtime client. The Azure Agent Client serves as the interface to interact with Azure OpenAI services. 

In [ ]:
agent_client = project_client.agents

print("Agent Client created successfully.")

## **Data Processing**

### Step 1: Convert Excel files to Markdown
Azure OpenAI Agents require data in specific file formats for processing. Since Excel files aren’t natively supported for File Search, our files need to be converted to Markdown tables in separate Markdown files. This step involves:

- Reading Excel files using pandas.
- Writing the data into Markdown format for compatibility.

In [ ]:
import pandas as pd
from pathlib import Path

# Define paths
output_dir = "uploads"
data_dir_path = "data"
sales_order_files = ["SalesOrder_43659.xlsx", "SalesOrder_43661.xlsx", "SalesOrder_43662.xlsx", "SalesOrder_43665.xlsx"]
output_dir_path = Path(data_dir_path) / output_dir

# Ensure output directory exists
if not Path(output_dir_path).exists():
    Path(output_dir_path).mkdir(parents=True, exist_ok=True)

# Prefix the data directory path to each file
sales_order_files = [Path(data_dir_path) / curr_file for curr_file in sales_order_files]

# Store the uploaded file IDs to be used later when enabling File Search
markdown_file_paths = []

with traced_span("convert_excels_to_markdown", file_count=len(sales_order_files)):
    for curr_file in sales_order_files:
        try:
            with traced_span("convert_single_file", file=str(curr_file)):
                # Check if the file exists
                if not Path(curr_file).exists():
                    raise FileNotFoundError(f"The file '{curr_file}' does not exist.")

                # Load the Excel File into a DataFrame
                df = pd.read_excel(curr_file)
                print(f"Workbook '{curr_file}' successfully loaded.")

                # Get the base name of the Excel file without the extension
                base_name = Path(curr_file).stem

                # Convert the DataFrame to a Markdown table
                md_tbl_str = df.to_markdown(index=False, tablefmt="pipe")

                # Define the output file name
                output_file = Path(output_dir_path) / f"{base_name}.md"
                markdown_file_paths.append(output_file)

                # Write the Markdown table to the file
                with Path(output_file).open("w") as f:
                    f.write(md_tbl_str)

                print(f"Markdown file '{output_file}' successfully written.")

        except FileNotFoundError as e:
            print(f"Error: {e}")

        except Exception as e:
            print(f"An error occurred while processing the Excel file '{curr_file}': {e}")

### Step 2: Upload Markdown file(s) to OpenAI
Next, we'll upload the markdown files from the local directory, `data\uploads`, to the Azure OpenAI Project Deployment. Before uploading, any markdown files in the project with the same name are removed to ensure the latest versions are used and to prevent duplicates. This step efficiently manages cleanup and file upload.

In [ ]:
uploaded_file_ids = []

try:
    with traced_span("upload_markdown_files", local_dir=str(output_dir_path)):
        for local_file in os.listdir(output_dir_path):
            # Define the local file path
            local_file_path = Path(output_dir_path) / local_file

            # Check if the file is a Markdown file
            if not local_file.endswith(".md"):
                continue
            else:
                print(f"Processing File '{local_file_path}'...")

                # Check if the file already exists in the cloud, and delete it if it does
                # for cloud_file in project_client.agents.list_files().data:
                for cloud_file in project_client.agents.files.list().data:
                    if cloud_file.filename == local_file:
                        project_client.agents.files.delete(cloud_file.id)
                        print(f"Deleted existing cloud file: '{cloud_file.filename}'")

                # Use the upload and poll SDK helper to upload the local file, add them to the vector store,
                # and poll the status of the file batch for completion.
                with traced_span("upload_single_file", filename=local_file):
                    file = project_client.agents.files.upload_and_poll(file_path=local_file_path, purpose="assistants")
                    uploaded_file_ids.append(file.id)
                    print(f"Uploaded file, file ID: {file.id}")

    print(f"File IDs: {uploaded_file_ids}")

except FileNotFoundError:
    print(f"Error: The file '{curr_file}' was not found.")
    raise

except Exception as e:
    print(f"An error occurred while processing the Excel file: {e}")
    raise

### Step 3: Create Vector Store
The Vector Store is used to store embeddings of uploaded files, enabling the File Search tool to efficiently locate relevant content.

In [ ]:
# Create a vector store called "Financial Statements"
with traced_span("create_vector_store", file_count=len(uploaded_file_ids)):
    vector_store = project_client.agents.vector_stores.create_and_poll(file_ids=uploaded_file_ids, name="Sales Orders")
    print(f"Vector store '{vector_store.name} ({vector_store.id})' created successfully.")

### Step 4: Create File Search Tool
The File Search tool enables the AI agent to query the uploaded files. It uses the embeddings stored in the Vector Store to locate and retrieve relevant content.

In [ ]:
from azure.ai.agents.models import FileSearchTool

# Create file search tool with resources followed by creating agent
with traced_span("create_file_search_tool", vector_store_id=getattr(vector_store, 'id', None)):
    file_search = FileSearchTool(vector_store_ids=[vector_store.id])
    print(f"File search tool created successfully for Vector Store {vector_store.name} ({vector_store.id}).")

## **Running the Azure OpenAI Agent**

### Step 1: Create a Sales Analyst Agent

In [ ]:
try:
    with traced_span("create_agent", model=__AZURE_OPENAI_DEPLOYMENT, name="Sales Analyst Agent"):
        agent = project_client.agents.create_agent(
            model=__AZURE_OPENAI_DEPLOYMENT,
            name="Sales Analyst Agent",
            instructions=(
                "You are an expert sales analyst. "
                "Use your knowledge base to answer questions about company sales, customers, and products."
            ),
            tools=file_search.definitions,
            tool_resources=file_search.resources,
        )
    print(f"Agent created successfully.({agent.id})")
except Exception as e:
    print("Error creating Agent:", e)

### Step 2: Start a New Converstaion
Conversation threads in Azure OpenAI enable context-aware interactions, storing both user prompts and agent responses. This step creates a new thread for the Sales Analyst Agent to handle queries.

In [ ]:
# Create a conversation thread
try:
    with traced_span("create_thread", agent_id=getattr(agent, 'id', None)):
        thread = agent_client.threads.create()
    print(f"Thread created successfully ({thread.id})")
except Exception as e:
    print("Error creating thread:", e)

### Step 3: Query the AI Agent
Next, we’ll add a user message to the thread. The AI agent responds to a user-defined prompt, such as calculating revenue by region. It processes the prompt and retrieves the necessary data to deliver insights.

In [ ]:
# Define the user question
prompt_content = "Which of the sales orders are not from US ?"

# Add the question to the thread
try:
    with traced_span("add_user_message", thread_id=getattr(thread, 'id', None), prompt_len=len(prompt_content)):
        message = agent_client.messages.create(
            thread_id=thread.id,
            role="user",
            content=prompt_content,
        )
    print(f"Successfully added User prompt to the thread. (Message ID {message.id})")
except Exception as e:
    print("Error adding user question:", e)

### Step 4: Run the AI Agent
In this step, we instruct the AI agent to process the user’s query within the created thread. The agent analyzes the context, executes the required tools (e.g., File Search), and generates a response. The `create_and_proces_run` function triggers the agent to process the user's prompt and produce an agent output.

In [ ]:
# Initiate the Agent's response
try:
    with traced_span("run_agent", agent_id=getattr(agent, 'id', None), thread_id=getattr(thread, 'id', None)):
        run = agent_client.runs.create_and_process(
            thread_id=thread.id,
            agent_id=agent.id,
        )
    print("Run started:", run.id)
except Exception as e:
    print("Error starting run:", e)

If the agent run fails, we will identify the issue. A common cause is exceeding the rate limit, requiring additional Azure quotas.

In [ ]:
if run.status == "failed":
    # Check if you got "Rate limit is exceeded.", then you want to get more quota
    print(f"Run failed: {run.last_error}")

### Step 5: Extract Insights
After the agent processes the prompt, we retrieve its responses, which contain the requested insights. This step ensures that the conversation thread is queried to fetch all relevant messages generated during the interaction.

In [ ]:
# Retrieves all messages from the thread in ascending order after the user message
with traced_span("list_messages", thread_id=getattr(thread, 'id', None)):
    messages = agent_client.messages.list(thread_id=thread.id, order="asc")

print("Run completed!\n\nMESSAGES\n")

# Loop through messages and print content based on role
for msg in messages:
    role = msg.role
    content = msg.content[0].text.value
    print(f"{role.capitalize()}: {content}")

## **Clean Up** - DO NOT TRIGGER TO MOVE FORWARD WITH EVALUATION SCENARIO

Properly cleaning up Azure resources after completing the analysis is essential to maintain a tidy Azure AI Studio Project environment and to avoid incurring unnecessary costs.

### Step 1: Deleting the Vector Store
Remove the Vector Store to free up storage resources.

In [ ]:
project_client.agents.vector_store_files.delete(vector_store.id, file_id=file.id)
print("Deleted vector store")

### Step 2: Deleting Uploaded Files
Ensure all uploaded files are removed from Azure to maintain a clean environment.


In [ ]:
# For each file id in the list, delete the file
for file_id in uploaded_file_ids:
    project_client.agents.files.delete(file_id)
    print(f"Deleted file: {file_id}")

### Step 3: Deleting the AI Agent
Remove the AI Agent to release associated resources.

In [ ]:
# Get agent list
agents = agent_client.list_agents()

# If the agent exists in the list, delete it
if agent.id in [a.id for a in agents]:
    response = agent_client.delete_agent(agent.id)
    print("Deleted Agent Client\n", response)
else:
    print("Agent does not exist to delete.")

### Step 4: Deleting local Markdown files
Remove the local Markdown files generated during this scenario to maintain a clean local environment.

In [ ]:
# Delete all local markdown files from the output directory
for file_path in markdown_file_paths:
    Path(file_path).unlink()
    print(f"Deleted local markdown file: {file_path}")

# Evaluate your generative AI application locally with the Azure AI Evaluation SDK

In [ ]:
import os, json, time
from pathlib import Path
from typing import List, Dict
from dotenv import load_dotenv

load_dotenv()

from azure.ai.evaluation import (
    evaluate,
    QAEvaluator,
    SimilarityEvaluator,
    RelevanceEvaluator,
)

from azure.ai.evaluation._model_configurations import AzureOpenAIModelConfiguration

with traced_span("evaluation_setup_start"):
    model_config = AzureOpenAIModelConfiguration(
        azure_endpoint="https://semantic-aifoundry.cognitiveservices.azure.com/",
        api_key=os.environ["AZURE_OPENAI_API_KEY"],
        api_version="2025-01-01-preview",
        azure_deployment="gpt-4o",
    )

    # Paths (robust across notebook working directories and OS)
    # Prefer ./data when running inside Sales_Analyst; fall back to repo root layout.
    candidate_dirs = [
        Path.cwd() / "data",
        Path.cwd() / "Sales_Analyst" / "data",
        Path("data"),
        Path("Sales_Analyst") / "data",
    ]
    DATA_DIR = next((p for p in candidate_dirs if p.exists()), None)
    if DATA_DIR is None:
        # As a fallback, create ./data relative to CWD
        DATA_DIR = Path.cwd() / "data"
        DATA_DIR.mkdir(parents=True, exist_ok=True)

    INPUT_JSONL = DATA_DIR / "sales_qa.jsonl"
    OUTPUT_JSONL = DATA_DIR / "sales_qa_with_responses.jsonl"
    EVAL_SUMMARY = DATA_DIR / "sales_qa_eval_results.json"

print(f"Working directory: {Path.cwd()}")
print(f"Resolved DATA_DIR: {DATA_DIR}")
print(f"Looking for dataset at: {INPUT_JSONL}")

if not INPUT_JSONL.exists():
    raise FileNotFoundError(f"Input dataset not found at: {INPUT_JSONL}. Please verify the path or create the dataset first.")

In [ ]:
# Define helper function to get agent response and derived context (citations)
def answer_with_agent(question: str) -> dict:
    try:
        with traced_span("eval_answer_with_agent", question_len=len(question)):
            # Create a new thread for this question
            with traced_span("eval_create_thread", agent_id=getattr(agent, 'id', None)):
                thread_local = agent_client.threads.create()
            
            # Add the question to the thread
            with traced_span("eval_add_message", thread_id=getattr(thread_local, 'id', None)):
                agent_client.messages.create(
                    thread_id=thread_local.id,
                    role="user",
                    content=question
                )
            
            # Run the agent
            with traced_span("eval_run_agent", agent_id=getattr(agent, 'id', None), thread_id=getattr(thread_local, 'id', None)):
                run_local = agent_client.runs.create_and_process(
                    thread_id=thread_local.id,
                    agent_id=agent.id
                )
            
            # Check if run failed
            if run_local.status == "failed":
                return {"response": f"<run_failed: {run_local.last_error}>", "context": ""}
            
            # Get messages and extract the assistant's response
            with traced_span("eval_list_messages", thread_id=getattr(thread_local, 'id', None)):
                messages_list = list(agent_client.messages.list(thread_id=thread_local.id, order="asc"))
            assistant_messages = [msg for msg in messages_list if getattr(msg.role, "value", str(msg.role)) == "assistant"]
            response_text = "<no_assistant_response>"
            derived_contexts = []

            # Build a map from file_id to filename to resolve citations (best-effort)
            file_name_map = {}
            try:
                for f in project_client.agents.files.list().data:
                    file_name_map[getattr(f, "id", None)] = getattr(f, "filename", None)
            except Exception:
                pass

            if assistant_messages:
                last_message = assistant_messages[-1]
                # Response text
                try:
                    if last_message.content and len(last_message.content) > 0 and getattr(last_message.content[0], "text", None):
                        response_text = last_message.content[0].text.value
                except Exception:
                    pass

                # Try to extract citations/annotations from all content parts
                try:
                    for part in getattr(last_message, "content", []):
                        txt = getattr(part, "text", None)
                        if txt is None:
                            continue
                        anns = getattr(txt, "annotations", None)
                        if not anns:
                            continue
                        for ann in anns:
                            # Different SDKs may surface file citations differently
                            file_id = None
                            quote = None
                            try:
                                file_citation = getattr(ann, "file_citation", None)
                                if file_citation is not None:
                                    file_id = getattr(file_citation, "file_id", None)
                                    quote = getattr(file_citation, "quote", None)
                            except Exception:
                                pass
                            # Fallbacks
                            if not file_id:
                                file_id = getattr(ann, "file_id", None)
                            if not quote:
                                quote = getattr(ann, "text", None)
                            fname = file_name_map.get(file_id) if file_id else None
                            snippet = quote or ""
                            if fname:
                                derived_contexts.append(f"[{fname}] {snippet}".strip())
                            elif snippet:
                                derived_contexts.append(snippet)
                except Exception:
                    pass

            context_text = "\n".join(dict.fromkeys([c for c in derived_contexts if c]))  # unique preserve order
            return {"response": response_text, "context": context_text}
    except Exception as e:
        return {"response": f"<error: {e}>", "context": ""}

In [ ]:
# 1) Load QA dataset and generate model responses
qa_rows: List[Dict] = []
with traced_span("evaluation_generate_responses"):
    with open(INPUT_JSONL, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            item = json.loads(line)
            question = item.get("question", "").strip()
            ground_truth = item.get("expected_answer", "").strip()
            if not question:
                continue
            # Measure latency of the agent answer
            t0 = time.time()
            result_obj = answer_with_agent(question)
            latency = time.time() - t0
            response = result_obj.get("response", "")
            derived_context = result_obj.get("context", "")
            response_len = len(response) if isinstance(response, str) else 0

            # Add a per-item span for observability
            with traced_span("evaluation_item", latency_ms=int(latency * 1000), response_length=response_len):
                qa_rows.append({
                    "query": question,
                    "ground_truth": ground_truth,
                    "response": response,
                    # Populate context from citations when available
                    "context": derived_context,
                    # Include latency (sec) and response_length (chars)
                    "latency": round(latency, 3),
                    "response_length": response_len,
                })

In [ ]:
with open(OUTPUT_JSONL, "w", encoding="utf-8") as f:
    for row in qa_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"Wrote {len(qa_rows)} rows with responses to: {OUTPUT_JSONL}")

In [ ]:
# 2) Run evaluations on the generated dataset
qa_eval = QAEvaluator(model_config)
sim_eval = SimilarityEvaluator(model_config)
rel_eval = RelevanceEvaluator(model_config)

with traced_span("evaluation_run", rows=len(qa_rows)):
    result = evaluate(
        data=str(OUTPUT_JSONL),
        evaluators={
            "qa": qa_eval,
            "similarity": sim_eval,
            "relevance": rel_eval,
        },
        evaluator_config={
            "default": {
                "column_mapping": {
                    "query": "${data.query}",
                    "response": "${data.response}",
                    "ground_truth": "${data.ground_truth}",
                    "context": "${data.context}",
                }
            }
        },
        azure_ai_project=os.environ.get("AZURE_AI_AGENT_ENDPOINT"), # Optional: submit to Azure AI Project if set
        output_path=str(EVAL_SUMMARY),
    )

# Print a compact summary
metrics = result.get("metrics", {})
print("\nEvaluation metrics (aggregate):")
for k, v in metrics.items():
    print(f"- {k}: {v}")

studio_url = result.get("studio_url")
if studio_url:
    print("\nView results in Azure AI Studio:", studio_url)

print(f"\nFull evaluation output saved to: {EVAL_SUMMARY}")

# Run the Evaluation in the Cloud

##### Upload the evaluation data to Azure AI

In [ ]:
import os
from pathlib import Path

# Dataset metadata (can be overridden via environment variables)
dataset_name ="sales-dataset"
dataset_version = "1.0"

# Use the JSONL with generated responses for cloud evaluation
data_path = OUTPUT_JSONL if 'OUTPUT_JSONL' in globals() else None
if data_path is None:
    raise RuntimeError("OUTPUT_JSONL wasn't set. Run the local generation step first.")

if not Path(data_path).exists():
    raise FileNotFoundError(f"Evaluation dataset not found: {data_path}. Generate it first in the previous section.")

# Upload a local JSONL file. Skip this step if you already have a dataset registered.
with traced_span("upload_eval_dataset", name=dataset_name, version=dataset_version):
    data_id = project_client.datasets.upload_file(
        name=dataset_name,
        version=dataset_version,
        file_path=str(data_path),
    ).id

print(f"Uploaded dataset '{dataset_name}:{dataset_version}' with id: {data_id}")

### RERUN THE EVALUATION WITH FLAG ENABLED `azure_ai_project=os.environ.get("AZURE_AI_AGENT_ENDPOINT")` TO SUBMIT AI FOUNDRY